# Cropify ML Model Training with K-Fold Cross-Validation
Phase 4: Optimize the Multilayer Perceptron
---
This notebook applies k-fold cross-validation to optimize the MLP model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
df = pd.read_csv('csv/Crop_recommendation.csv')
print(f"Dataset shape: {df.shape}")
print(f"Unique crops: {df['label'].nunique()}")
df.head()

## Preprocessing

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data scaled successfully")

## K-Fold Cross-Validation for MLP

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hidden_layer_configs = [
    (50,),
    (100,),
    (100, 50),
    (128, 64),
    (128, 64, 32),
    (256, 128),
]

results = []
for hidden_layers in hidden_layer_configs:
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        activation='relu'
    )
    scores = cross_val_score(mlp, X_scaled, y_encoded, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()
    results.append({
        'hidden_layers': str(hidden_layers),
        'mean_cv_score': mean_score,
        'std_cv_score': std_score,
        'fold_scores': scores
    })
    print(f"{hidden_layers}: {mean_score:.4f} (+/- {std_score:.4f})")

## Select Best Configuration

In [ ]:
best_config = max(results, key=lambda x: x['mean_cv_score'])
print(f"Best configuration: {best_config['hidden_layers']}")
print(f"Best CV Score: {best_config['mean_cv_score']:.4f}")

## Train Final Model with Best Config

In [ ]:
best_layers = eval(best_config['hidden_layers'])
final_mlp = MLPClassifier(
    hidden_layer_sizes=best_layers,
    max_iter=2000,
    random_state=42,
    early_stopping=True,
    activation='relu',
    learning_rate='adaptive',
    learning_rate_init=0.001
)
final_mlp.fit(X_scaled, y_encoded)

joblib.dump(final_mlp, 'MLP.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("Model and scaler saved!")

## Grid Search for Further Optimization

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'hidden_layer_sizes': [best_layers],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01],
}

grid_search = GridSearchCV(
    MLPClassifier(max_iter=1000, random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_scaled, y_encoded)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

## Save Optimized Model

In [ ]:
optimized_mlp = grid_search.best_estimator_
joblib.dump(optimized_mlp, 'MLP.pkl')
print("Optimized MLP saved!")